# 02. Double Machine Learning (DML) & Robustness Checks
**Team Member**: Causal Modeler (Member 2)
**Objective**: Fit first-stage nuisance models, estimate the treatment effect (ATE & ITE) with cross-fitting, and perform placebo & covariate balance checks.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data.synthetic import generate_synthetic_data
from src.models.dml_estimator import DoubleMLEstimator
from src.models.robustness import run_placebo_test, check_covariate_balance

# Load data
df = generate_synthetic_data(n_samples=5000, random_state=42)
feature_cols = ["Age", "PriorSpend", "VisitFrequency", "LoyaltyScore"]
X = df[feature_cols]
D = df["Discount"]
Y = df["Sales"]
print(f"Loaded {len(df)} samples for causal modeling.")

Loaded 5000 samples for causal modeling.


In [2]:
# Covariate balance check prior to modeling
balance_df = check_covariate_balance(X, D, feature_cols)
print(balance_df)

          Feature  Treated_Mean  Control_Mean     SMD  Balanced
0             Age        37.280        39.062  0.1565     False
1      PriorSpend        95.021       132.712  0.3443     False
2  VisitFrequency         3.729         4.251  0.2671     False
3    LoyaltyScore         2.680         2.930  0.1604     False


In [3]:
# Fit Double Machine Learning Estimator
dml = DoubleMLEstimator(n_splits=5, nuisance_type="random_forest", backend="native")
dml.fit(X, D, Y)
print(f"Estimated ATE: {dml.ate_:.3f} +/- {1.96 * dml.standard_error_:.3f}")

[2026-10-21 15:54:40] [INFO] [DoubleMLEstimator]: Fitting DML Estimator using backend 'native' and 'random_forest' nuisances...


[2026-10-21 15:54:43] [INFO] [DoubleMLEstimator]: Native DML fit complete: ATE = 31.329 +/- 1.131


Estimated ATE: 31.329 +/- 1.131


In [4]:
# Robustness Check: Placebo Test with permuted treatment
placebo_results = run_placebo_test(DoubleMLEstimator, X, D, Y, n_iterations=3)
print("Placebo Test Outcome:", placebo_results)

[2026-10-21 15:54:43] [INFO] [RobustnessChecks]: Running 3 iterations of placebo test (randomized treatment)...


[2026-10-21 15:54:43] [INFO] [DoubleMLEstimator]: Fitting DML Estimator using backend 'native' and 'random_forest' nuisances...


[2026-10-21 15:54:44] [INFO] [DoubleMLEstimator]: Native DML fit complete: ATE = -0.549 +/- 1.396


[2026-10-21 15:54:44] [INFO] [DoubleMLEstimator]: Fitting DML Estimator using backend 'native' and 'random_forest' nuisances...


[2026-10-21 15:54:46] [INFO] [DoubleMLEstimator]: Native DML fit complete: ATE = 0.995 +/- 1.394


[2026-10-21 15:54:46] [INFO] [DoubleMLEstimator]: Fitting DML Estimator using backend 'native' and 'random_forest' nuisances...


[2026-10-21 15:54:48] [INFO] [DoubleMLEstimator]: Native DML fit complete: ATE = -0.541 +/- 1.391


[2026-10-21 15:54:48] [INFO] [RobustnessChecks]: Placebo Test: Mean Effect = -0.0314 (std=0.7261) | Passed: True


Placebo Test Outcome: {'passed': True, 'mean_placebo_ate': -0.031428416413131655, 'std_placebo_ate': 0.726086841715124, 'iterations': [-0.5485434192174092, 0.9954045616077877, -0.5411463916297734]}
